In [34]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path("__file__").resolve().parents[1]
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch')

In [224]:
def collect(state: str = "FL", per_page: int = 100) -> pd.DataFrame:
    fields = ",".join([
        "id",
        "school.name",
        "school.city",

        "location.lat",
        "location.lon",

        "latest.programs.cip_4_digit.code",
        "latest.programs.cip_4_digit.unit_id",
        "latest.programs.cip_4_digit.title",
        "latest.programs.cip_4_digit.school.type",
        "latest.programs.cip_4_digit.credential.level",
        "latest.programs.cip_4_digit.distance",

        "latest.school.locale",
        "latest.school.carnegie_size_setting",
        "latest.admissions.admission_rate.overall",
        "latest.student.demographics.median_family_income",
        "latest.student.students_with_pell_grant",
        "latest.school.open_admissions_policy",
        "latest.student.demographics.age_entry",
        "latest.school.title_iv.eligibility_type",
        "latest.programs.cip_4_digit.earnings.1_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.1_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.2_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.2_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.3_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.3_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.4_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.4_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.5_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.5_yr.working_not_enrolled.overall_count"
    ])

    params = {
        "api_key": SCORECARD_KEY,
        "school.state": state,
        "fields": fields,
        "per_page": str(per_page),

        # only programs with earnings reported
        "latest.programs.cip_4_digit.earnings.4_yr.overall_median_earnings__range": "1.."
    }

    dfs = []

    params["page"] = "0"
    response = get_with_retries(BASE_URL, params=params, timeout=30)
    data = get_json_or_raise(response)

    total = int(data["metadata"]["total"])
    per_page_actual = int(data["metadata"]["per_page"])
    total_pages = math.ceil(total / per_page_actual)

    META = [
        "id",
        "school.name",
        "school.city",
        "location.lat",
        "location.lon",
        "latest.school.locale",
        "latest.school.carnegie_size_setting",
        "latest.admissions.admission_rate.overall",
        "latest.student.demographics.median_family_income",
        "latest.student.students_with_pell_grant",
        "latest.school.open_admissions_policy",
        "latest.student.demographics.age_entry",
        "latest.school.title_iv.eligibility_type",
    ]

    def page_to_df(data):
        results = [r for r in data.get("results", []) if r.get("latest.programs.cip_4_digit")]
        return pd.json_normalize(
            results,
            record_path=["latest.programs.cip_4_digit"],
            meta=META,
            errors="ignore",
        )

    dfs.append(page_to_df(data))

    for page in range(1, total_pages):
        params["page"] = str(page)
        response = get_with_retries(BASE_URL, params=params, timeout=30)
        data = get_json_or_raise(response)
        dfs.append(page_to_df(data))

    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    print(f"Total pages fetched: {total_pages}")
    print(f"Total Rows/Programs ingested: {len(df)}")
    return df

def get_json_or_raise(response: requests.Response):
    # Raise for HTTP errors early (4xx/5xx)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        ct = response.headers.get("Content-Type", "")
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"HTTP {response.status_code} for {response.url}\n"
            f"Content-Type: {ct}\n"
            f"Body preview:\n{body_preview}"
        ) from e

    # Check content-type sanity (helps catch HTML responses)
    ct = response.headers.get("Content-Type", "")
    if "json" not in ct.lower():
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"Expected JSON but got Content-Type: {ct}\n"
            f"URL: {response.url}\n"
            f"Body preview:\n{body_preview}"
        )

    # Parse JSON with a clearer error if it fails
    try:
        return response.json()
    except json.JSONDecodeError as e:
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"JSON decode failed for {response.url}\n"
            f"Body preview:\n{body_preview}"
        ) from e
    
    import time
import random
import requests

def get_with_retries(url, params, tries=5, timeout=30):
    last = None
    for i in range(tries):
        r = requests.get(url, params=params, timeout=timeout, headers={"Accept": "application/json"})
        if r.status_code < 500:
            return r
        last = r
        time.sleep((2 ** i) + random.random())
    return last

# Removed features
---
### 1

```
"latest.student.demographics.avg_family_income"
"latest.student.demographics.median_hh_income"
```
overlaps with ```"latest.student.demographics.median_family_income"```

---
### 2

```
"latest.academics.program_reporter.programs_offered"
```
~83% is null

---
### 3

```
latest.admissions.test_requirements
```
~46% is null and overlaps with ```latest.admissions.admission_rate.overall  ```

---
### 4

```
"latest.admissions.sat_scores.50th_percentile.critical_reading",
"latest.admissions.sat_scores.50th_percentile.math",
"latest.admissions.act_scores.50th_percentile.cumulative",
"latest.admissions.act_scores.50th_percentile.english",
"latest.admissions.act_scores.50th_percentile.math",
"latest.admissions.sat_scores.average.overall",
"latest.admissions.act_scores.midpoint.cumulative"
```
~58% is null and overlaps with ```latest.admissions.admission_rate.overall``` and ```latest.school.open_admissions_policy```



In [225]:
tdf = collect()
# tdf = pd.read_csv(root/"data"/"clean"/"scorecard"/"clean_ml_scorecard_FL_programs.csv")
display(tdf.head())
display(tdf.info())

Total pages fetched: 3
Total Rows/Programs ingested: 2501


,code,title,unit_id,distance,school.type,credential.level,earnings.1_yr.overall_median_earnings,earnings.1_yr.working_not_enrolled.overall_count,earnings.4_yr.overall_median_earnings,earnings.4_yr.working_not_enrolled.overall_count,earnings.5_yr.overall_median_earnings,earnings.5_yr.working_not_enrolled.overall_count,id,school.name,school.city,location.lat,location.lon,latest.school.locale,latest.school.carnegie_size_setting,latest.admissions.admission_rate.overall,latest.student.demographics.median_family_income,latest.student.students_with_pell_grant,latest.school.open_admissions_policy,latest.student.demographics.age_entry,latest.school.title_iv.eligibility_type
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,25586.0,22.0,22265,28,33286.0,21.0,132374,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,29493.0,32.0,41177,27,NaN,NaN,132374,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,36966.0,63.0,47325,48,NaN,NaN,132374,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,34269.0,24.0,42839,28,NaN,NaN,132374,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,32976.0,33.0,33081,29,NaN,NaN,132374,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,None,16748,None,1,26,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 25 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   code                                              2501 non-null   object 
 1   title                                             2501 non-null   object 
 2   unit_id                                           2501 non-null   int64  
 3   distance                                          2501 non-null   int64  
 4   school.type                                       2501 non-null   object 
 5   credential.level                                  2501 non-null   int64  
 6   earnings.1_yr.overall_median_earnings             2100 non-null   float64
 7   earnings.1_yr.working_not_enrolled.overall_count  2100 non-null   float64
 8   earnings.4_yr.overall_median_earnings             2501 non-null   int64  
 9   earnings.4_yr.worki

None

# Why so many missing Admission Rates?

In [227]:
df = tdf.copy()
round(df.isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.4226

In [228]:
round(df[df["latest.school.open_admissions_policy"]==2].isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.023

Approximately 40% of institutions have missing values for admission_rate.overall.
According to IPEDS reporting rules, institutions with an open admissions policy do not report traditional selectivity metrics such as admission rate or standardized test scores.

A check confirms that nearly all non-open-admission institutions report admission rates, indicating that the missingness is structural rather than random.

Therefore, missing admission rates are interpreted as corresponding primarily to open-admission institutions, and the open_admissions_policy variable is retained to preserve this structural distinction.

In [229]:
df = df.drop(columns="id")
df = clean(df)

Numeric columns: Index(['distance', 'credential_level', '1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count', 'admission_rate_overall',
       'median_family_income', 'students_with_pell_grant'],
      dtype='object')


C:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\src\ira\clean\clean_scorecard.py:7: FutureWarning:

The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.



In [230]:
df["selectivity_bucket"] = pd.cut(
    df["admission_rate_overall"],
    bins=[0, 0.3, 0.7, 1],
    labels=["elite", "mid", "open"]
)

In [231]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   code                       2501 non-null   string  
 1   title                      2501 non-null   string  
 2   unit_id                    2501 non-null   string  
 3   distance                   2501 non-null   int64   
 4   school_type                2501 non-null   string  
 5   credential_level           2501 non-null   int64   
 6   1_yr_median_earnings       2100 non-null   float64 
 7   1_yr_working_count         2100 non-null   float64 
 8   4_yr_median_earnings       2501 non-null   int64   
 9   4_yr_working_count         2501 non-null   int64   
 10  5_yr_median_earnings       1788 non-null   float64 
 11  5_yr_working_count         1788 non-null   float64 
 12  school_name                2501 non-null   string  
 13  school_city                2501 n

None

In [232]:
df.head()

,code,title,unit_id,distance,school_type,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,school_name,school_city,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,25586.0,22.0,22265,28,33286.0,21.0,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,29493.0,32.0,41177,27,NaN,NaN,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,36966.0,63.0,47325,48,NaN,NaN,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,34269.0,24.0,42839,28,NaN,NaN,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,32976.0,33.0,33081,29,NaN,NaN,Atlantic Technical College,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN


In [233]:
save(clean(df),"clean","ml_scorecard_FL_programs.csv")
save(tdf,file_name="ml_scorecard_FL_programs.csv")

Numeric columns: Index(['distance', 'credential_level', '1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count', 'admission_rate_overall',
       'median_family_income', 'students_with_pell_grant'],
      dtype='object')


C:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\src\ira\clean\clean_scorecard.py:7: FutureWarning:

The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.



In [234]:
display("Relevant numeric variables statistics:", df.select_dtypes(exclude=object).describe())
display("Missing values per column:", df.isna().sum())
display("Correlation matrix:", df.corr(numeric_only=True))

'Relevant numeric variables statistics:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,admission_rate_overall,median_family_income,students_with_pell_grant
count,2501.000000,2501.000000,2100.000000,2100.000000,2501.000000,2501.000000,1788.000000,1788.000000,1444.000000,2491.000000,2303.000000
mean,1.431028,4.064774,47436.760952,135.764286,62386.475010,132.365054,60389.043624,140.090604,0.535130,29262.739864,0.734936
std,0.829784,10.392411,22519.842497,364.461526,25274.413965,382.210450,25962.542895,356.876270,0.228989,13060.807166,0.127915
min,0.000000,1.000000,11191.000000,16.000000,10611.000000,16.000000,12010.000000,16.000000,0.189000,0.000000,0.192053
25%,1.000000,2.000000,31025.000000,29.000000,46115.000000,26.000000,43159.750000,29.000000,0.401100,21349.000000,0.667383
50%,1.000000,3.000000,43529.500000,53.000000,57448.000000,46.000000,55379.000000,55.000000,0.546600,25118.000000,0.721905
75%,2.000000,3.000000,59504.250000,126.250000,75539.000000,111.000000,74449.250000,134.000000,0.696600,38662.000000,0.829670
max,3.000000,99.000000,246053.000000,10218.000000,221571.000000,9437.000000,271873.000000,6833.000000,1.000000,81806.000000,0.990401


'Missing values per column:'

code                            0
title                           0
unit_id                         0
distance                        0
school_type                     0
credential_level                0
1_yr_median_earnings          401
1_yr_working_count            401
4_yr_median_earnings            0
4_yr_working_count              0
5_yr_median_earnings          713
5_yr_working_count            713
school_name                     0
school_city                     0
location_lat                    0
location_lon                    0
locale                          0
carnegie_size_setting           0
admission_rate_overall       1057
median_family_income           10
students_with_pell_grant      198
open_admissions_policy          4
age_entry                      10
title_iv_eligibility_type       0
selectivity_bucket           1057
dtype: int64

'Correlation matrix:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,admission_rate_overall,median_family_income,students_with_pell_grant
distance,1.000000,-0.153944,0.103553,0.056294,0.092568,0.073129,0.063412,0.074980,0.132306,-0.054622,0.056554
credential_level,-0.153944,1.000000,0.547171,-0.025134,0.048096,-0.029798,0.593513,-0.061411,-0.217481,0.005684,0.049736
1_yr_median_earnings,0.103553,0.547171,1.000000,0.046434,0.910971,0.012900,0.868921,-0.015934,-0.084154,0.196585,-0.180965
1_yr_working_count,0.056294,-0.025134,0.046434,1.000000,0.042067,0.934041,0.051325,0.774639,0.054556,-0.051221,0.026993
4_yr_median_earnings,0.092568,0.048096,0.910971,0.042067,1.000000,0.025095,0.935040,-0.013464,-0.215881,0.333229,-0.311550
4_yr_working_count,0.073129,-0.029798,0.012900,0.934041,0.025095,1.000000,0.026617,0.825968,0.058224,-0.059164,0.050633
5_yr_median_earnings,0.063412,0.593513,0.868921,0.051325,0.935040,0.026617,1.000000,-0.008875,-0.207934,0.332783,-0.320055
5_yr_working_count,0.074980,-0.061411,-0.015934,0.774639,-0.013464,0.825968,-0.008875,1.000000,0.056293,-0.091484,0.072359
admission_rate_overall,0.132306,-0.217481,-0.084154,0.054556,-0.215881,0.058224,-0.207934,0.056293,1.000000,-0.416269,0.393917
median_family_income,-0.054622,0.005684,0.196585,-0.051221,0.333229,-0.059164,0.332783,-0.091484,-0.416269,1.000000,-0.905237


In [183]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [239]:
model_df = df.copy()
target = '4_yr_median_earnings'

drop_columns=["title","4_yr_working_count","school_name",'1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count']
X = model_df.drop(columns=[target]).copy()
X = X.drop(columns=[],errors="ignore")
y = pd.to_numeric(model_df[target], errors='coerce').copy()

cat_cols = [
    'code',
    'school_type',
    'locale',
    'carnegie_size_setting',
    'open_admissions_policy',
    'title_iv_eligibility_type',
    'credential_level',
    'distance',
    'school_city',
    "selectivity_bucket"
]
num_cols = [
    # 'admission_rate_overall',
    "location_lat",
    "location_lon", 
    'median_family_income',
    'students_with_pell_grant',
    'age_entry'
    
]

# categorical: force plain object and replace missing with np.nan
for c in cat_cols:
    X[c] = X[c].astype(object)
    X[c] = X[c].replace({pd.NA: np.nan})

# numeric: force numeric with np.nan
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

# nuclear option: remove any lingering pd.NA anywhere in X
X = X.astype(object).replace({pd.NA: np.nan})

# now restore numeric cols back to numeric dtype
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

In [252]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', Ridge())
])

param_grid = {
    'reg__alpha': [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

y_train_log=np.log(y_train)
y_test_log=np.log(y_test)

grid.fit(X_train, y_train_log)

print("Best params:", grid.best_params_)
print("Best CV MAE:", round(-grid.best_score_, 2))
best_model = grid.best_estimator_
preds = best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, preds), 2))
print("Test R2:", round(r2_score(y_test_log, preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'reg__alpha': 1.0}
Best CV MAE: 0.14
Test MAE: 0.14
Test R2: 0.7669


In [253]:
ls_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Lasso(max_iter=100000, random_state=42))
])

ls_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

ls_grid = GridSearchCV(
    ls_pipe,
    param_grid=ls_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ls_grid.fit(X_train, y_train_log)

print("Best params:", ls_grid.best_params_)
print("Best CV MAE:", round(-ls_grid.best_score_, 2))

ls_best_model = ls_grid.best_estimator_
ls_preds = ls_best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, ls_preds), 2))
print("Test R2:", round(r2_score(y_test_log, ls_preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'reg__alpha': 0.01}
Best CV MAE: 0.24
Test MAE: 0.23
Test R2: 0.4663


In [249]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=100000))
])


enet_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "reg__l1_ratio": [0.005, 0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    enet_pipe,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(X_train, y_train_log)

print("ElasticNet Best params:", enet_grid.best_params_)
print("ElasticNet Best CV MAE:", round(-enet_grid.best_score_, 2))
enet_best_model = enet_grid.best_estimator_
enet_preds = enet_best_model.predict(X_test)

print("ElasticNet Test MAE:", round(mean_absolute_error(y_test_log, enet_preds), 2))
print("ElasticNet Test R2:", round(r2_score(y_test_log, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
ElasticNet Best params: {'reg__alpha': 0.01, 'reg__l1_ratio': 0.005}
ElasticNet Best CV MAE: 0.17
ElasticNet Test MAE: 0.17
ElasticNet Test R2: 0.682


In [241]:
baseline_pred = [y_train_log.mean()] * len(y_test_log)

print("Baseline MAE:", round(mean_absolute_error(y_test_log, baseline_pred), 2))

Baseline MAE: 0.31


In [243]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict


pipe_rfr = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

param_grid_rfr = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 5, 10, 20],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4]
}

grid_rfr = GridSearchCV(
    estimator=pipe_rfr,
    param_grid=param_grid_rfr,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid_rfr.fit(X_train, y_train_log)

print("RF Best params:", grid_rfr.best_params_)
print("RF Best CV MAE:", round(-grid_rfr.best_score_, 2))

best_rfr = grid_rfr.best_estimator_
rfr_preds = best_rfr.predict(X_test)

print("RF Test MAE:", round(mean_absolute_error(y_test_log, rfr_preds), 2))
print("RF Test R2:", round(r2_score(y_test_log, rfr_preds), 4))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
RF Best params: {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 5, 'clf__n_estimators': 100}
RF Best CV MAE: 0.15
RF Test MAE: 0.15
RF Test R2: 0.7273


In [244]:
from xgboost import XGBRegressor

pipe_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', XGBRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ))
])

param_grid_xgb = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 5, 10, 20],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__subsample': [0.8, 1.0]
}

grid_xgb_log = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid_xgb_log.fit(X_train, y_train_log)

print("XGB Best params:", grid_xgb_log.best_params_)
print("XGB Best CV MAE:", round(-grid_xgb_log.best_score_, 2))

best_xgb_log = grid_xgb_log.best_estimator_
xgb_preds_log = best_xgb_log.predict(X_test)

print("XGB Test MAE:", round(mean_absolute_error(y_test_log, xgb_preds_log), 2))
print("XGB Test R2:", round(r2_score(y_test_log, xgb_preds_log), 4))

Fitting 5 folds for each of 72 candidates, totalling 360 fits
XGB Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 10, 'clf__n_estimators': 300, 'clf__subsample': 0.8}
XGB Best CV MAE: 0.13
XGB Test MAE: 0.13
XGB Test R2: 0.7838


Fitting 5 folds for each of 72 candidates, totalling 360 fits
XGB Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 10, 'clf__n_estimators': 300, 'clf__subsample': 0.8}
XGB Best CV MAE: 0.13
XGB Test MAE: 0.13
XGB Test R2: 0.7833

All models significantly outperformed the baseline. The linear model and Random Forest performed similarly, suggesting that linear relationships explain a large portion of the variance. However, XGBoost achieved the best performance, reducing MAE from 0.14 to 0.13 and increasing R2 to ~0.78. This indicates that nonlinear interactions exist in the data and are effectively captured by gradient boosting methods.

In [245]:
feature_names = best_xgb_log.named_steps["preprocessor"].get_feature_names_out()

importances = best_xgb_log.named_steps["clf"].feature_importances_

importance = pd.Series(importances, index=feature_names)

print("most important features")
display(importance.sort_values(ascending=False).head(15))
print("least important features")
display(importance.sort_values(ascending=True).head(15))

most important features


cat__credential_level_1          0.114552
cat__code_1204                   0.066623
cat__open_admissions_policy_2    0.047120
cat__code_5138                   0.030520
cat__credential_level_7          0.019006
cat__credential_level_5          0.015057
cat__credential_level_6          0.013698
cat__code_1409                   0.012667
cat__code_5005                   0.012660
cat__selectivity_bucket_elite    0.012549
cat__code_1101                   0.011720
cat__code_1410                   0.011284
cat__code_5009                   0.011088
cat__code_5135                   0.010505
cat__code_5007                   0.010345
dtype: float32

least important features


cat__school_city_Plantation       0.0
cat__school_city_Ave Maria        0.0
cat__school_city_Ormond Beach     0.0
cat__school_city_Avon Park        0.0
cat__school_city_Babson Park      0.0
cat__school_city_Niceville        0.0
cat__school_city_Boynton Beach    0.0
cat__school_city_Chipley          0.0
cat__code_1203                    0.0
cat__school_city_Miramar          0.0
cat__school_city_Coconut Creek    0.0
cat__school_city_Pensacola        0.0
cat__school_city_Miami Lakes      0.0
cat__school_city_Marianna         0.0
cat__school_city_Delray Beach     0.0
dtype: float32

Feature importance analysis from the XGBoost model shows that categorical variables, particularly program codes (CIP), credential level, admission policy, and carnegie classification, were the most influential predictors. Additionally, several features had zero importance, indicating that certain categories did not contribute meaningfully to prediction, likely due to lack of signal.

In [254]:
cv_preds = cross_val_predict(grid_xgb_log, X, np.log(y), cv=cv, method="predict")

log_error_df = X.copy()
log_error_df["4_year_earning"] = np.log(y)
log_error_df["4_year_pred"] = cv_preds
# log_error_df["error"] = log_error_df["actual"] - log_error_df["pred"]

cols_to_add = ["title", "school_name", "credential_level","4_yr_working_count"]
log_error_df[cols_to_add] = model_df.loc[log_error_df.index, cols_to_add]

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits


The model was trained on log-transformed earnings to address skewness and improve predictive stability. Predictions were then exponentiated back to the original scale to evaluate performance and interpret errors in dollar terms.

Predictions are the typical (median-like) expected earnings rather than the average, which is appropriate given the skewed nature of income data.

In [ ]:
error_df=log_error_df.copy()
display(error_df.head())

delog_col = ["4_year_earning","4_year_pred"]
for col in delog_col:
    error_df[col] = np.exp(error_df[col])

error_df["4_year_error"] = error_df["4_year_earning"] - error_df["4_year_pred"]

display(error_df.head())

,code,title,unit_id,distance,school_type,credential_level,school_city,location_lat,location_lon,locale,carnegie_size_setting,median_family_income,students_with_pell_grant,age_entry,title_iv_eligibility_type,actual,pred,school_name,4_yr_working_count
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,10.010771,10.265214,Atlantic Technical College,28
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,10.625635,10.553132,Atlantic Technical College,27
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,10.764794,10.596947,Atlantic Technical College,48
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,10.665204,10.819522,Atlantic Technical College,28
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,10.406714,10.445498,Atlantic Technical College,29


,code,title,unit_id,distance,school_type,credential_level,school_city,location_lat,location_lon,locale,carnegie_size_setting,median_family_income,students_with_pell_grant,age_entry,title_iv_eligibility_type,actual,pred,school_name,4_yr_working_count,error
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,22265.0,28716.123047,Atlantic Technical College,28,-6451.123047
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,41177.0,38297.199219,Atlantic Technical College,27,2879.800781
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,47325.0,40012.480469,Atlantic Technical College,48,7312.519531
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,42839.0,49987.183594,Atlantic Technical College,28,-7148.183594
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,Coconut Creek,26.24278,-80.192271,21,-2,16748.0,NaN,26.0,1,33081.0,34389.222656,Atlantic Technical College,29,-1308.222656


In [255]:
def run_earnings_model(df, year, random_state=42, cv=5):
    year = int(year)
    target = f"{year}_yr_median_earnings"

    # columns from other horizons to drop
    other_year_cols = []
    for y in [1, 4, 5]:
        if y != year:
            other_year_cols.extend([
                f"{y}_yr_median_earnings",
                f"{y}_yr_working_count",
            ])

    model_df = df.copy()

    # clean target
    model_df[target] = pd.to_numeric(model_df[target], errors="coerce")
    model_df = model_df[model_df[target].notna() & (model_df[target] > 0)].copy()

    # build X / y
    drop_columns = ["title", "school_name"] + other_year_cols

    X = model_df.drop(columns=drop_columns, errors="ignore").copy()
    X = X.drop(columns=[target], errors="ignore")
    y = pd.to_numeric(model_df[target], errors="coerce").copy()
    y_log = np.log(y)

    # feature groups
    cat_cols = [
        "code",
        "school_type",
        "locale",
        "carnegie_size_setting",
        "open_admissions_policy",
        "title_iv_eligibility_type",
        "credential_level",
        "distance",
        "school_city",
        "selectivity_bucket",
    ]

    num_cols = [
        "location_lat",
        "location_lon",
        "median_family_income",
        "students_with_pell_grant",
        "age_entry",
    ]

    cat_cols = [c for c in cat_cols if c in X.columns]
    num_cols = [c for c in num_cols if c in X.columns]

    # coerce dtypes
    for c in cat_cols:
        X[c] = X[c].astype(object).replace({pd.NA: np.nan})

    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X = X.astype(object).replace({pd.NA: np.nan})

    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # split
    X_train, X_test, y_train_log, y_test_log = train_test_split(
        X, y_log, test_size=0.2, random_state=random_state
    )

    # preprocessing
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(missing_values=np.nan, strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(missing_values=np.nan, strategy="median")),
        ("scaler", StandardScaler()),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ])

    # model
    pipe_xgb = Pipeline([
        ("preprocessor", preprocessor),
        ("reg", XGBRegressor(
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )),
    ])

    param_grid_xgb = {
        "reg__n_estimators": [100, 200, 300],
        "reg__max_depth": [3, 5, 10],
        "reg__learning_rate": [0.01, 0.05, 0.1],
        "reg__subsample": [0.8, 1.0],
    }

    grid_xgb_log = GridSearchCV(
        estimator=pipe_xgb,
        param_grid=param_grid_xgb,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=1,
        refit=True,
        verbose=1,
        error_score="raise",
    )

    # fit
    grid_xgb_log.fit(X_train, y_train_log)

    print(f"\n===== {year}-YEAR MODEL =====")
    print("XGB Best params:", grid_xgb_log.best_params_)
    print("XGB Best CV MAE (log):", round(-grid_xgb_log.best_score_, 4))

    best_xgb_log = grid_xgb_log.best_estimator_
    xgb_preds_log = best_xgb_log.predict(X_test)

    print("XGB Test MAE (log):", round(mean_absolute_error(y_test_log, xgb_preds_log), 4))
    print("XGB Test R2 (log):", round(r2_score(y_test_log, xgb_preds_log), 4))

    # cross-validated preds on all rows
    cv_preds_log = cross_val_predict(
        best_xgb_log,
        X,
        y_log,
        cv=cv,
        method="predict",
        n_jobs=1,
    )

    # dynamic column names
    actual_log_col = f"{year}_year_earning_log"
    pred_log_col = f"{year}_year_pred_log"
    actual_col = f"{year}_year_earning"
    pred_col = f"{year}_year_pred"
    error_col = f"{year}_year_error"

    error_df = X.copy()
    error_df[actual_log_col] = y_log
    error_df[pred_log_col] = cv_preds_log

    # add metadata back
    cols_to_add = [c for c in ["title", "school_name", "credential_level"] if c in model_df.columns]
    error_df[cols_to_add] = model_df.loc[error_df.index, cols_to_add]

    # delog dynamically
    delog_col_map = {
        actual_log_col: actual_col,
        pred_log_col: pred_col,
    }

    for log_col, out_col in delog_col_map.items():
        error_df[out_col] = np.exp(error_df[log_col])

    error_df[error_col] = error_df[actual_col] - error_df[pred_col]

    return {
        "year": year,
        "target": target,
        "grid": grid_xgb_log,
        "best_model": best_xgb_log,
        "X": X,
        "y": y,
        "error_df": error_df,
    }

In [265]:
results_1 = run_earnings_model(df, year=1, random_state=RANDOM_STATE, cv=cv)
results_4 = run_earnings_model(df, year=4, random_state=RANDOM_STATE, cv=cv)
results_5 = run_earnings_model(df, year=5, random_state=RANDOM_STATE, cv=cv)

Fitting 5 folds for each of 54 candidates, totalling 270 fits

===== 1-YEAR MODEL =====
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE (log): 0.1501
XGB Test MAE (log): 0.1277
XGB Test R2 (log): 0.8566
Fitting 5 folds for each of 54 candidates, totalling 270 fits

===== 4-YEAR MODEL =====
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE (log): 0.1318
XGB Test MAE (log): 0.1302
XGB Test R2 (log): 0.7838
Fitting 5 folds for each of 54 candidates, totalling 270 fits

===== 5-YEAR MODEL =====
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE (log): 0.1394
XGB Test MAE (log): 0.1175
XGB Test R2 (log): 0.8644


In [ ]:
display(results_1["error_df"].head())
display(results_4["error_df"].head())
display(results_5["error_df"].head())

,code,unit_id,distance,school_type,credential_level,1_yr_working_count,school_city,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,1_year_earning_log,1_year_pred_log,title,school_name,1_year_earning,1_year_pred,1_year_error
0,1205,132374,1,Public,1,22.0,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.149801,9.945742,Culinary Arts and Related Services.,Atlantic Technical College,25586.0,20863.191406,4722.808594
1,4603,132374,2,Public,1,32.0,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.291908,10.388206,Electrical and Power Transmission Installers.,Atlantic Technical College,29493.0,32474.373047,-2981.373047
2,4702,132374,1,Public,1,63.0,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.517754,10.522847,"Heating, Air Conditioning, Ventilation and Ref...",Atlantic Technical College,36966.0,37154.757812,-188.757813
3,4706,132374,1,Public,1,24.0,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.441996,10.381479,Vehicle Maintenance and Repair Technologies/Te...,Atlantic Technical College,34269.0,32256.642578,2012.357422
4,5108,132374,1,Public,1,33.0,Coconut Creek,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.403535,10.268359,Allied Health and Medical Assisting Services.,Atlantic Technical College,32976.0,28806.583984,4169.416016


,code,unit_id,distance,school_type,credential_level,5_yr_working_count,school_city,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error
0,1205,132374,1,Public,1,21.0,Coconut Creek,26.242780,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.412892,9.914482,Culinary Arts and Related Services.,Atlantic Technical College,33286.0,20221.105469,13064.894531
8,1101,132471,2,"Private, nonprofit",3,68.0,Miami,25.878908,-80.198925,21,13,0.7724,28822.0,0.712681,2,26.0,1,open,11.029520,11.288042,"Computer and Information Sciences, General.",Barry University,61668.0,79860.937500,-18192.937500
9,1304,132471,2,"Private, nonprofit",5,86.0,Miami,25.878908,-80.198925,21,13,0.7724,28822.0,0.712681,2,26.0,1,open,11.266628,11.206766,Educational Administration and Supervision.,Barry University,78169.0,73626.937500,4542.062500
10,1313,132471,2,"Private, nonprofit",5,46.0,Miami,25.878908,-80.198925,21,13,0.7724,28822.0,0.712681,2,26.0,1,open,11.119542,11.007211,Teacher Education and Professional Development...,Barry University,67477.0,60307.441406,7169.558594
11,2201,132471,1,"Private, nonprofit",7,204.0,Miami,25.878908,-80.198925,21,13,0.7724,28822.0,0.712681,2,26.0,1,open,11.262154,11.460504,Law.,Barry University,77820.0,94892.835938,-17072.835938


In [205]:
error_df=error_df[error_df["4_yr_working_count"] >= 20]

valid_codes = (
    error_df.groupby("code")["school_name"]
    .nunique()
)

valid_codes = valid_codes[valid_codes >= 4].index
error_df = error_df[error_df["code"].isin(valid_codes)]

error_df["school_count"] = error_df.groupby(["code","credential_level"])["school_name"].transform("nunique")

error_df["confidence"] = pd.cut(
    error_df["school_count"],
    bins=[0, 5, 15, 100],
    labels=["low", "medium", "high"]
)

error_df["pct_error"] = error_df["error"] / error_df["pred"]

k = np.percentile(np.log1p(error_df["4_yr_working_count"]), 75)

weight = np.log1p(error_df["4_yr_working_count"])/np.log1p(error_df["4_yr_working_count"]+k)

error_df["score"] = (
    error_df["pct_error"] * weight
)

In [206]:
np.percentile(np.log1p(error_df["4_yr_working_count"]), [25, 50, 75, 90, 95])

array([3.4657359 , 4.04305127, 4.87519732, 5.71373281, 6.14825234])

In [207]:
rank_pct = error_df["pct_error"].rank()
rank_score = error_df["score"].rank()

adjustments = (rank_pct - rank_score).abs().mean()
adjustments/len(error_df)

0.002346400981878565

In [208]:
error_df["rank_diff"] = (rank_score - rank_pct)

# display(error_df.sort_values("score").head(10))
error_df.sort_values("rank_diff").tail(10)
error_df.sort_values("rank_diff").head(10)

,code,title,unit_id,distance,school_type,credential_level,school_city,location_lat,location_lon,locale,carnegie_size_setting,median_family_income,students_with_pell_grant,age_entry,title_iv_eligibility_type,actual,pred,school_name,4_yr_working_count,error,school_count,confidence,pct_error,score,rank_diff
217,5106,Dental Support Services and Allied Professions.,133386,1,Public,2,Daytona Beach,29.202780,-81.049513,13,12,21285.0,0.853585,25.0,1,63800.0,55871.195312,Daytona State College,20,7928.804687,10,medium,0.141912,0.132806,-30.0
1739,1312,Teacher Education and Professional Development...,137476,3,"Private, nonprofit",5,Miami Gardens,25.922071,-80.253273,21,13,26170.0,0.770297,24.0,1,64254.0,59194.984375,St. Thomas University,20,5059.015625,7,medium,0.085464,0.079979,-27.0
1386,1108,Computer Software and Media Applications.,136358,1,Public,2,Lake Worth,26.612560,-80.086530,21,15,21408.0,0.921804,23.0,1,48988.0,45146.867188,Palm Beach State College,20,3841.132812,4,low,0.085081,0.079621,-24.0
249,0901,Communication and Media Studies.,133492,1,"Private, nonprofit",3,Saint Petersburg,27.715798,-82.687043,11,11,77695.0,0.411392,21.0,1,60969.0,53399.222656,Eckerd College,22,7569.777344,15,medium,0.141758,0.133569,-23.0
1635,5138,"Registered Nursing, Nursing Administration, Nu...",137315,1,Public,3,Avon Park,27.569740,-81.508667,23,9,20658.0,0.955259,23.0,1,102066.0,89269.148438,South Florida State College,25,12796.851562,57,high,0.143351,0.136169,-20.0
1801,5109,"Allied Health Diagnostic, Intervention, and Tr...",137759,0,Public,99,Tallahassee,30.443866,-84.339000,12,12,23457.0,0.817773,22.0,1,58514.0,51028.292969,Tallahassee State College,20,7485.707031,4,low,0.146697,0.137284,-20.0
1140,0183,Veterinary/Animal Health Technologies/Technici...,135717,1,Public,2,Miami,25.777717,-80.190856,11,15,18391.0,0.967728,22.0,1,48841.0,46270.066406,Miami Dade College,20,2570.933594,4,low,0.055564,0.051998,-19.0
1967,4805,Precision Metal Working.,138497,1,Public,1,Inverness,28.834861,-82.345563,23,-2,17183.0,NaN,28.0,1,45279.0,39535.085938,Withlacoochee Technical College,22,5743.914063,15,medium,0.145286,0.136893,-19.0
2134,1304,Educational Administration and Supervision.,433660,3,Public,5,Fort Myers,26.463638,-81.772600,21,16,43546.0,0.598170,21.0,1,73910.0,69206.546875,Florida Gulf Coast University,20,4703.453125,20,high,0.067963,0.063601,-19.0
1045,1304,Educational Administration and Supervision.,135081,3,"Private, nonprofit",6,Fort Lauderdale,26.185869,-80.163959,12,15,21659.0,0.786796,28.0,1,80618.0,74474.296875,Keiser University-Ft Lauderdale,21,6143.703125,5,low,0.082494,0.077477,-19.0


“The weighting scheme introduces only minor adjustments to ranking positions, suggesting that prediction error remains dominant while program size provides a secondary refinement.”

To avoid instability in groups with small sample sizes, a small constant (epsilon) was added to the standard deviation when computing the final score. This prevents artificially inflated scores caused by near-zero variance estimates, while still allowing all groups to be included in the analysis.

To account for differences in sample size, a soft penalization factor was applied using sqrt(n / (n + k)). This approach reduces the influence of groups with small sample sizes without excluding them entirely. As n increases, the penalty diminishes, allowing larger groups to retain their full weight while appropriately down-weighting less reliable estimates.

In [209]:
error_df=error_df.drop(columns="rank_diff")

In [212]:
program_variability = (
    error_df
    .groupby(["code","credential_level"])
    .agg(
        std_score=("score", "std"),
        mean_score=("score", "mean"),
        min_score=("score", "min"),
        max_score=("score", "max"),
        n=("score", "size")
    )
)

epsilon=0.064

program_variability["relative_variability"] = (
    program_variability["std_score"]/(program_variability["mean_score"].abs()+epsilon)
)
program_variability["rv_weighted"] = (
    (program_variability["relative_variability"])
    * (program_variability["n"]/(program_variability["n"]+10)) 
)

top_programs=program_variability.sort_values("rv_weighted", ascending=False).head(10)
titles = error_df[["code", "credential_level", "title"]].drop_duplicates()

top_programs = (
    program_variability
    .sort_values("rv_weighted", ascending=False)
    .head(50)
    .reset_index()
    .merge(titles, on=["code", "credential_level"], how="left")
)
top_programs.head(10)

,code,credential_level,std_score,mean_score,min_score,max_score,n,relative_variability,rv_weighted,title
0,5107,2,0.384277,0.020521,-0.229840,1.353466,15,4.546529,2.727917,Health and Medical Administrative Services.
1,1204,1,0.180960,0.014944,-0.444767,0.584115,81,2.292270,2.040372,Cosmetology and Related Personal Grooming Serv...
2,5122,5,0.268009,-0.008087,-0.356428,0.609156,11,3.717835,1.947437,Public Health.
3,5135,1,0.199836,0.012441,-0.457123,0.378330,27,2.614239,1.907688,Somatic Bodywork and Related Therapeutic Servi...
4,5009,3,0.257739,0.009539,-0.466666,0.529087,10,3.504818,1.752409,Music.
5,5108,2,0.176199,-0.019738,-0.527841,0.398975,23,2.104184,1.466553,Allied Health and Medical Assisting Services.
6,4706,1,0.208194,0.037340,-0.407240,0.483146,24,2.054416,1.450176,Vehicle Maintenance and Repair Technologies/Te...
7,5110,1,0.264019,0.017139,-0.280808,0.561967,8,3.253911,1.446183,Clinical/Medical Laboratory Science/Research a...
8,5202,2,0.155190,0.014184,-0.307995,0.254021,26,1.984920,1.433553,"Business Administration, Management and Operat..."
9,5107,1,0.155972,-0.014985,-0.362253,0.295664,25,1.974722,1.410516,Health and Medical Administrative Services.


In [216]:
(
    error_df[(error_df["code"] == "5135")&(error_df["credential_level"]==1)]
    .sort_values(["credential_level", "score"], ascending=[True, False])
)

,code,title,unit_id,distance,school_type,credential_level,school_city,location_lat,location_lon,locale,carnegie_size_setting,median_family_income,students_with_pell_grant,age_entry,title_iv_eligibility_type,actual,pred,school_name,4_yr_working_count,error,school_count,confidence,pct_error,score
2304,5135,Somatic Bodywork and Related Therapeutic Servi...,451149,1,"Private, for-profit",1,Davie,26.070447,-80.251766,21,-2,20646.0,0.765051,25.0,1,38981.0,28048.396484,Aveda Institute-South Florida,40,10932.603516,25,high,0.389776,0.378330
1368,5135,Somatic Bodywork and Related Therapeutic Servi...,136303,1,Public,1,Orlando,28.550422,-81.383783,11,-2,14956.0,NaN,26.0,1,37194.0,27359.804688,Orange Technical College,24,9834.195312,25,high,0.359440,0.340589
890,5135,Somatic Bodywork and Related Therapeutic Servi...,134228,0,"Private, for-profit",1,Fort Pierce,27.411962,-80.325377,21,-2,11766.0,0.956923,28.0,1,31517.0,25428.703125,Fort Pierce Beauty Academy,31,6088.296875,25,high,0.239426,0.230015
2173,5135,Somatic Bodywork and Related Therapeutic Servi...,438285,1,"Private, for-profit",1,Maitland,28.629921,-81.397179,21,-2,17597.0,0.770259,29.0,1,30699.0,25114.238281,Cortiva Institute,293,5584.761719,25,high,0.222374,0.221733
2043,5135,Somatic Bodywork and Related Therapeutic Servi...,387925,1,"Private, for-profit",1,Pompano Beach,26.275351,-80.150541,13,-2,17597.0,0.770259,29.0,1,30699.0,25802.457031,Cortiva Institute,293,4896.542969,25,high,0.189770,0.189223
2273,5135,Somatic Bodywork and Related Therapeutic Servi...,449490,1,"Private, for-profit",1,West Palm Beach,26.639480,-80.115507,21,-2,15750.0,0.732308,31.0,1,32263.0,27388.390625,Palm Beach Academy of Health & Beauty,47,4874.609375,25,high,0.177981,0.173642
2198,5135,Somatic Bodywork and Related Therapeutic Servi...,443234,1,"Private, for-profit",1,Margate,26.243429,-80.199738,21,-2,12879.0,0.964706,28.0,1,31519.0,27201.648438,Hollywood Institute,54,4317.351562,25,high,0.158717,0.155423
2047,5135,Somatic Bodywork and Related Therapeutic Servi...,406149,1,"Private, for-profit",1,Sarasota,27.389606,-82.485901,21,-2,17000.0,0.769634,33.0,1,29432.0,26012.369141,Sarasota School of Massage Therapy,94,3419.630859,25,high,0.131462,0.130033
1986,5135,Somatic Bodywork and Related Therapeutic Servi...,362700,1,"Private, for-profit",1,Fort Myers,26.596006,-81.870312,13,-2,16015.0,0.893788,28.0,1,32463.0,28693.648438,Aveda Institute-Fort Myers,54,3769.351562,25,high,0.131365,0.128639
2118,5135,Somatic Bodywork and Related Therapeutic Servi...,430582,1,"Private, for-profit",1,Miami,25.756567,-80.367780,21,-2,4823.0,0.930318,34.0,1,27609.0,24968.468750,Praxis Institute,229,2640.531250,25,high,0.105755,0.105348


In [217]:
import plotly.express as px
bar_df = error_df[
    (error_df["code"] == "5122") & (error_df["credential_level"] == 5)
]

bar_df = bar_df.sort_values(by="error", ascending=True)

px.bar(
    bar_df,
    x="error",
    y="school_name",
    orientation="h",
    labels={"error":"Schools","school_name":"Pred. Error"},
    title="Master's in Public Health"
)

Programs such as Public Health show substantial variability in outcomes across institutions, even after controlling for observable factors. This suggests that institutional effects such as program quality, networking opportunities, or industry connections may play a significant role in shaping student outcomes.

In [219]:
save(error_df,file_name="residual_FL_programs.csv")